# Customer Churn Prediction — بخش اکتشافی (EDA)

این نوت‌بوک قسمت علمی پروژه است: داده‌ها را بررسی می‌کنیم، تمیز می‌کنیم، چند ویژگی جدید می‌سازیم و مدل‌ها را سریع مقایسه می‌کنیم.

نکته مهم: منطق اصلی پروژه در پکیج `src` است و این نوت‌بوک فقط از همان توابع استفاده می‌کند تا نتیجه‌ی اینجا با نتیجه‌ی داشبورد یکسان باشد.

نوت‌بوک را از ریشه پروژه اجرا کنید یا سلول زیر `sys.path` را تنظیم می‌کند.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # اگر نوت‌بوک از پوشه خودش اجرا شود

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_loader import load_dataset, dataset_summary
from src.preprocessing import clean_dataset, build_preprocessor
from src.feature_engineering import add_features

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 25)

## ۱) بارگذاری و نگاه اولیه به داده‌ها

In [ ]:
df = load_dataset('../data/dataset.csv')
df.head()

In [ ]:
summary = dataset_summary(df)
print(f"rows: {summary['rows']}, columns: {summary['columns']}")
print(f"missing cells: {summary['missing_cells']}, duplicates: {summary['duplicate_rows']}")
print()
print('ستون‌هایی که مقدار گم‌شده دارند:')
pd.Series(summary['missing_by_column'])[lambda s: s > 0]

در داده‌های تلکامی معمولاً `TotalCharges` برای مشتری‌های تازه‌وارد (با `tenure = 0`) خالی است. این خطای اندازه‌گیری نیست، بلکه یعنی مشتری هنوز صورت‌حسابی نداشته؛ پس بعداً به‌جای حذف، مقدار صفر می‌دهیم.

## ۲) توزیع کلاس هدف (Churn)

کلاس churn معمولاً اقلیت است. اگر داده خیلی نامتوازن باشد، Accuracy به‌تنهایی گمراه‌کننده می‌شود و باید به Precision/Recall/ROC-AUC نگاه کنیم.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x=config.TARGET_COLUMN, hue=config.TARGET_COLUMN, order=['No', 'Yes'],
              hue_order=['No', 'Yes'], palette=['#22c55e', '#ef4444'], legend=False, ax=axes[0])
axes[0].set_title('Churn distribution')

rates = df[config.TARGET_COLUMN].value_counts(normalize=True)
axes[1].bar(rates.index, rates.values, color=['#22c55e', '#ef4444'])
axes[1].set_title('Class share')
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

print(f"churn rate: {(df[config.TARGET_COLUMN] == 'Yes').mean():.1%}")

## ۳) رابطه ویژگی‌های عددی با Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(data=df, x='tenure', hue=config.TARGET_COLUMN, hue_order=['No', 'Yes'],
             palette=['#22c55e', '#ef4444'], kde=True, ax=axes[0])
axes[0].set_title('Tenure by churn')
sns.histplot(data=df, x='MonthlyCharges', hue=config.TARGET_COLUMN, hue_order=['No', 'Yes'],
             palette=['#22c55e', '#ef4444'], kde=True, ax=axes[1])
axes[1].set_title('Monthly charges by churn')
plt.tight_layout()
plt.show()

الگوی کلاسیک این داده‌ها: مشتری‌های تازه‌وارد (tenure کم) و کسانی که قرارداد ماه‌به‌ماه دارند بیشتر لغو می‌کنند، و مشتریان قدیمی با قرارداد دوساله تقریباً پایدارند.

In [ ]:
# churn به تفکیک نوع قرارداد — معمولاً قوی‌ترین سیگنال داده است
pd.crosstab(df['Contract'], df[config.TARGET_COLUMN], normalize='index').round(3)

## ۴) همبستگی ویژگی‌های عددی

In [ ]:
numeric = df[['tenure', 'MonthlyCharges', 'SeniorCitizen']].copy()
numeric['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
numeric['Churn'] = (df[config.TARGET_COLUMN] == 'Yes').astype(int)

plt.figure(figsize=(6, 4.5))
sns.heatmap(numeric.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation matrix (numeric features)')
plt.tight_layout()
plt.show()

همبستگی بالای `tenure` و `TotalCharges` بدیهی است (هرچه مدت اشتراک بیشتر، مجموع پرداختی بیشتر). این یعنی هم‌خطی؛ برای مدل‌هایی مثل رگرسیون لجستیک با تنظیم `C` مدیریت می‌شود و برای مدل‌های درختی/همسایه اساساً مسئله‌ساز نیست.

## ۵) تمیزکاری و مهندسی ویژگی (با همان توابع `src`)

- `clean_dataset`: تبدیل `TotalCharges` به عدد، پر کردن مشتری‌های صفر‌ماهه با 0 و یکدست‌سازی رشته‌ها.
- `add_features`: ساخت `ServicesCount`، `HasSupport`، `AvgMonthlyCharge` و `TenureGroup` فقط از روی ویژگی‌های خام (بدون نشت اطلاعات).

In [ ]:
from sklearn.model_selection import train_test_split

cleaned = clean_dataset(df).drop_duplicates()
featured = add_features(cleaned)

X = featured.drop(columns=[config.TARGET_COLUMN])
y = featured[config.TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, stratify=y, random_state=config.RANDOM_STATE
)
print('train:', X_train.shape, '| test:', X_test.shape)

## ۶) اعتبارسنجی متقاطع مدل‌ها با پیش‌پردازش داخل Pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from src.models import build_models
from src.validation import cross_validate_pipeline

numeric_features = config.RAW_NUMERIC + config.ENGINEERED_NUMERIC
categorical_features = config.RAW_CATEGORICAL
preprocessor = build_preprocessor(numeric_features, categorical_features)

rows = []
for name, estimator in build_models().items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    cv = cross_validate_pipeline(pipe, X_train, y_train)
    rows.append({
        'Model': name,
        'Accuracy': f"{cv['accuracy']['mean']:.3f} ± {cv['accuracy']['std']:.3f}",
        'F1': f"{cv['f1']['mean']:.3f} ± {cv['f1']['std']:.3f}",
        'ROC-AUC': f"{cv['roc_auc']['mean']:.3f} ± {cv['roc_auc']['std']:.3f}",
    })
pd.DataFrame(rows)

## ۷) ارزیابی نهایی بهترین مدل روی تست

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from src.evaluation import full_test_evaluation

best_pipe = Pipeline([('preprocess', preprocessor), ('model', LogisticRegression(max_iter=1000))])
best_pipe.fit(X_train, y_train)

evaluation = full_test_evaluation(best_pipe, X_test, y_test)
print(pd.Series(evaluation['metrics']).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
fpr, tpr = evaluation['roc_curve']
axes[0].plot(fpr, tpr)
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title(f"ROC curve (AUC = {evaluation['metrics']['roc_auc']:.3f})")

sns.heatmap(evaluation['confusion_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['No churn', 'Churn'], yticklabels=['No churn', 'Churn'], cbar=False)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

## جمع‌بندی

- داده‌ها تمیز شدند و چهار ویژگی مهندسی‌شده بدون نشت اطلاعات اضافه شد.
- هر سه مدل با ۵-فولد استراتیفاید اعتبارسنجی شدند؛ رگرسیون لجستیک معمولاً بهترین تعادل دقت/سرعت را دارد.
- اجرای کامل با GridSearch و مقایسه نهایی در داشبورد (`python app.py`) انجام می‌شود و نتیجه در `models/churn_model.joblib` ذخیره می‌شود.